In [ ]:
!pip install altair vega_datasets ipywidgets


In [ ]:
import pandas as pd
import altair as alt
from ipywidgets import widgets, interact

# Load and clean data
df = pd.read_csv('us_foreign_aid_funding.csv')
df = df[['Country Name', 'Funding Account Name', 'Transaction Type Name', 'Fiscal Year', 'current_amount']]
df = df[df['current_amount'] > 0]

# Assign regions
def assign_region(country):
    europe_eurasia = ['Albania', 'Armenia', 'Azerbaijan', 'Belarus', 'Bosnia and Herzegovina',
                      'Bulgaria', 'Croatia', 'Czechia', 'Estonia', 'Georgia', 'Hungary',
                      'Kazakhstan', 'Kosovo', 'Moldova', 'Montenegro', 'North Macedonia',
                      'Russia', 'Serbia', 'Ukraine']
    east_asia_oceania = ['China', 'Indonesia', 'Cambodia', 'Timor-Leste']
    south_central_asia = ['Afghanistan', 'Bangladesh', 'India', 'Pakistan']
    if country in europe_eurasia:
        return 'Europe & Eurasia'
    elif country in east_asia_oceania:
        return 'East Asia & Oceania'
    elif country in south_central_asia:
        return 'South & Central Asia'
    return None

df['Region'] = df['Country Name'].apply(assign_region)
df = df[df['Region'].notna()]

# Assign sectors
def assign_sector(name):
    name = str(name).lower()
    if 'health' in name or 'medical' in name or 'hiv' in name:
        return 'Health'
    if 'education' in name:
        return 'Education'
    if 'capital' in name or 'infrastructure' in name:
        return 'Infrastructure'
    if 'security' in name or 'defense' in name:
        return 'Security'
    if 'democracy' in name or 'governance' in name:
        return 'Democracy & Governance'
    return 'Other'

df['Sector'] = df['Funding Account Name'].apply(assign_sector)

# Aggregate
agg_df = df.groupby(['Region', 'Sector', 'Fiscal Year'])['current_amount'].sum().reset_index()

# Widget selectors
years = sorted(df['Fiscal Year'].unique())
year_selector = widgets.Dropdown(options=years, value=2023, description='Year:')
view_selector = widgets.RadioButtons(
    options=['absolute', 'percentage'],
    value='absolute',
    description='View:'
)

# Chart builder function
def create_chart(selected_year, view_type):
    data = agg_df[agg_df['Fiscal Year'] == selected_year].copy()

    if view_type == 'percentage':
        data['total'] = data.groupby('Region')['current_amount'].transform('sum')
        data['value'] = (data['current_amount'] / data['total']) * 100
        y_title = 'Percentage (%)'
    else:
        data['value'] = data['current_amount']
        y_title = 'Amount (USD)'

    chart = alt.Chart(data).mark_bar().encode(
        x=alt.X('Region:N', sort='-y'),
        y=alt.Y('value:Q', title=y_title),
        color='Sector:N',
        tooltip=['Sector', 'Region', 'current_amount']
    ).properties(
        title=f'U.S. Foreign Aid by Sector - {selected_year}',
        width=700,
        height=400
    ).interactive()

    return chart

# Display
@interact(year=year_selector, view_type=view_selector)
def show_chart(year, view_type):
    return create_chart(year, view_type)
